# aiswakepy — Run Pipeline

Edit the paths and parameters in **Cell 1** before running.  
Each cell is independent — re-run any stage without restarting the kernel.

In [ ]:
# === EDIT THESE PATHS FOR YOUR RUN ===
ais_csv          = r"data/compare_JI/ais/DT_JI_ChannelEdited.csv"
coastline_shp    = r"data/compare_JI/coastline/JI_shipwake_land_mask.shp"
bathy_file       = r"data/compare_JI/bathymetry/JI_bathy.dfsu"
tide_dfs0        = r"data/compare_JI/tide/JI_tide_2023.dfs0"
tide_item        = "OSSI: Surface elevation"   # item name inside the .dfs0 file
study_area_shp   = None   # optional polygon shapefile to clip study area, or None

# === OUTPUT FILENAMES (change to save different runs side by side) ===
output_dir           = r"data/compare_JI/output/"
wave_height_map_name = "WaveHeightMap.png"
wave_period_map_name = "WavePeriodMap.png"
wave_params_name     = "wave_params.csv"
shore_impact_name    = "shore_impact.csv"
save_stage_csv       = True   # save {ais_stem}_filtered/depth/wave/impact.csv as checkpoints

# === AIS FILTER PARAMETERS ===
traj_gap_s           = 180.0   # split trajectory on gap > this (seconds)
max_velocity_knots   = 12.0    # kinematic check: flag segment if avg speed exceeds this
max_acceleration_ms2 = 0.2     # acceleration check: replace SOG/COG if implied accel exceeds this
interp_interval_s    = 30.0    # Hermite spline interpolation time step (seconds)

# === WAVE CALCULATION PARAMETERS ===
formula        = "kriebel"  # empirical wake model: 'bhowmik', 'blaauw', 'gates', 'kriebel', 'maynord', 'pianc', 'sorensen'
cb_method      = "B_Le"     # block coefficient method: 'L_Le' (Kriebel default), 'B_Le', or 'table'
                            #   L_Le: Cb estimated from waterline length / bow entry length ratio
                            #   B_Le: uses beam instead of length; table: lookup by vessel type
gravity        = 9.78       # local gravitational acceleration (m/s²); Singapore value (vs 9.81 standard)
max_prop_m     = 2000.0     # maximum wake ray propagation distance from vessel track (m);
                            #   rays beyond this are discarded before coastline intersection
wake_cutoff_m  = 0.01       # minimum shore wave height to record (m); events below this are dropped

# === GENERAL WAKE FILTER PARAMETERS (applied regardless of formula) ===
max_sog_knots  = 12.0   # maximum vessel speed (knots)        (fast craft outside calculation range)
max_bl_ratio   = 0.3    # maximum beam / length ratio         (atypical hull form, model not valid)

# === KRIEBEL (2005) APPLICABILITY LIMITS (formula-specific; ignored for other formulas) ===
min_Froude_M   = 0.1    # minimum modified Froude number Fr*  (vessel must be in wave-making regime)
max_Froude_M   = 0.5    # maximum modified Froude number Fr*  (avoids supercritical planing regime)
max_bf         = 0.4    # maximum BF = Beta * (Fr*-0.1)^2     (no data in Kriebel's dataset exceeds 0.4)

# === OSSI COMPARISON PARAMETERS (Cell 9) ===
ossi_xlsx       = r"data/compare_JI/OSSI/ShipWake_peaks_event3minutes_minHeight5cm.xlsx"   # OSSI wave gauge events
gauge_lon       = 103.733335   # OSSI gauge longitude (decimal degrees)
gauge_lat       = 1.265771     # OSSI gauge latitude  (decimal degrees)
event_window    = 0.5          # matching window (minutes) for pairing AIS arrivals <-> OSSI events
ossi_cb_method  = "B_Le"       # block coefficient method used in the MATLAB comparison

In [ ]:
# === Cell 2: Build config ===
from aiswakepy.config import load_config

config = load_config({
    "ais": {
        "raw_csv": ais_csv,
        "traj_gap_s": traj_gap_s,
        "max_velocity_knots": max_velocity_knots,
        "max_acceleration_ms2": max_acceleration_ms2,
        "interp_interval_s": interp_interval_s,
        "study_area_shp": study_area_shp,
    },
    "vessel": {"cb_method": cb_method},
    "bathymetry": {
        "source": bathy_file,
        "tide_dfs0": tide_dfs0,
        "tide_item": tide_item,
    },
    "coastline": {"shapefile": coastline_shp},
    "wave": {
        "gravity": gravity,
        "min_Froude_M": min_Froude_M,
        "max_Froude_M": max_Froude_M,
        "max_bf": max_bf,
        "max_sog_knots": max_sog_knots,
        "max_bl_ratio": max_bl_ratio,
    },
    "impact": {"max_propagation_m": max_prop_m, "wake_cutoff_m": wake_cutoff_m},
    "output": {
        "directory": output_dir,
        "wave_height_map_name": wave_height_map_name,
        "wave_period_map_name": wave_period_map_name,
        "wave_params_name": wave_params_name,
        "shore_impact_name": shore_impact_name,
        "save_stage_csv": save_stage_csv,
    },
})
print("Config loaded OK")

In [3]:
# === Cell 3: Stage 1 — AIS Filter ===
from pathlib import Path
from aiswakepy.stages.filter import filter_ais

df_filtered = filter_ais(
    csv_path=config.ais.raw_csv,
    coastline_shp=config.coastline.shapefile,
    gap_s=config.ais.traj_gap_s,
    max_velocity_knots=config.ais.max_velocity_knots,
    max_acceleration_ms2=config.ais.max_acceleration_ms2,
    interval_s=config.ais.interp_interval_s,
    study_area_shp=config.ais.study_area_shp,
)
print(f"Filtered and interpolated: {len(df_filtered):,} rows")
if config.output.save_stage_csv:
    _stem = Path(config.ais.raw_csv).stem
    _out = Path(config.output.directory)
    _out.mkdir(parents=True, exist_ok=True)
    df_filtered.to_csv(_out / f"{_stem}_01_filtered.csv", index=False)
    print(f"  saved {_stem}_01_filtered.csv")
df_filtered.head()

Filtered and interpolated: 64,338 rows
  saved DT_JI_ChannelEdited_01_filtered.csv


,mmsi,width,length,draught,obstime,longitude,latitude,sog,cog,typecargo,segment_id
0,209459000,23.0,148.0,9.0,2023-03-05 11:32:03,103.737883,1.262588,12.200000,233.300000,70,1
1,209459000,23.0,148.0,9.0,2023-03-05 11:32:33,103.736510,1.261559,12.494430,233.022086,70,1
2,209459000,23.0,148.0,9.0,2023-03-05 11:33:03,103.735112,1.260505,12.697798,233.022382,70,1
3,209459000,23.0,148.0,9.0,2023-03-05 11:33:33,103.733694,1.259442,12.810096,233.287646,70,1
4,209459000,23.0,148.0,9.0,2023-03-05 11:34:03,103.732261,1.258383,12.832146,233.816540,70,1


In [4]:
# === Cell 4: Stage 2 — Depth + Tide Assignment ===
from aiswakepy.stages.depth import assign_depth

df_depth = assign_depth(
    df=df_filtered,
    bathy_path=config.bathymetry.source,
    tide_dfs0_path=config.bathymetry.tide_dfs0,
    tide_item=config.bathymetry.tide_item,
    underkeel_margin_m=config.bathymetry.underkeel_margin_m,
)
print(f"After depth filter: {len(df_depth)} rows")
if config.output.save_stage_csv:
    df_depth.to_csv(_out / f"{_stem}_02_depth.csv", index=False)
    print(f"  saved {_stem}_02_depth.csv")
df_depth[["WaterDepth", "draught"]].describe()

After depth filter: 61050 rows
  saved DT_JI_ChannelEdited_02_depth.csv


,WaterDepth,draught
count,61050.000000,61050.000000
mean,18.776709,5.592007
std,5.102648,2.723986
min,5.445060,1.000000
25%,13.757343,4.000000
50%,20.033738,5.000000
75%,22.561232,7.000000
max,31.385574,20.000000


In [ ]:
# === Cell 5: Stage 3 — Vessel Parameters ===
from aiswakepy.stages.vessel import compute_vessel_params

print(
    f"Vessel parameters  cb_method: {cb_method}\n"
    f"  general filters  SOG <= {max_sog_knots} kn  B/L <= {max_bl_ratio}"
)

df_vessel = compute_vessel_params(
    df=df_depth,
    cb_method=config.vessel.cb_method,
    g=config.wave.gravity,
    max_sog_knots=config.wave.max_sog_knots,
    max_bl_ratio=config.wave.max_bl_ratio,
)
print(f"Vessel events: {len(df_vessel)}")
if config.output.save_stage_csv:
    df_vessel.to_csv(_out / f"{_stem}_03_vessel.csv", index=False)
    print(f"  saved {_stem}_03_vessel.csv")
df_vessel[["SOGms", "Froude_D", "T", "Theta", "Cel", "Tc", "BLratio"]].describe()

In [ ]:
# === Cell 6: Stage 4 — Wave Impact ===
from aiswakepy.stages.wave_impact import compute_wave_impact

print(
    f"Wave impact  formula: {formula}\n"
    f"  Kriebel limits  Froude_M = [{min_Froude_M}, {max_Froude_M}]  BF <= {max_bf}"
)

df_wave_impact = compute_wave_impact(
    df_vessel=df_vessel,
    coastline_shp=config.coastline.shapefile,
    formula=formula,
    max_propagation_m=config.impact.max_propagation_m,
    wake_cutoff_m=config.impact.wake_cutoff_m,
    g=config.wave.gravity,
    rho=config.wave.rho_water,
    # Kriebel-specific applicability limits (ignored for other formulas)
    min_Froude_M=config.wave.min_Froude_M,
    max_Froude_M=config.wave.max_Froude_M,
    max_bf=config.wave.max_bf,
)
print(f"Shore impact events: {len(df_wave_impact)}")
if config.output.save_stage_csv:
    df_wave_impact.to_csv(_out / f"{_stem}_04_wave_impact.csv", index=False)
    print(f"  saved {_stem}_04_wave_impact.csv")
df_wave_impact.head()

In [7]:
# === Cell 7: Visualisation ===
from pathlib import Path
from aiswakepy.viz.wave_map import plot_wave_height_map, plot_wave_period_map

out = Path(output_dir)
out.mkdir(parents=True, exist_ok=True)

plot_wave_height_map(df_wave_impact, coastline_shp, out / wave_height_map_name, max_points=100_000)
plot_wave_period_map(df_wave_impact, coastline_shp, out / wave_period_map_name, max_points=100_000)
print(f"Maps saved to {out.resolve()}")

Maps saved to C:\Projects\61803960 SCDF Metocean (Sub)\Tools\PROJ-61803960-AISWAKEPY-WUHL\data\compare_JI\output


In [8]:
# === Cell 8: Export final results ===
# Stage CSVs are already saved above.
# This cell saves final outputs using the configured filenames.
from pathlib import Path

out = Path(output_dir)
df_vessel.to_csv(out / wave_params_name, index=False)
df_wave_impact.to_csv(out / shore_impact_name, index=False)
print(f"Results saved to {out.resolve()}")

Results saved to C:\Projects\61803960 SCDF Metocean (Sub)\Tools\PROJ-61803960-AISWAKEPY-WUHL\data\compare_JI\output


## Cell 9 — Compare All Formulae to OSSI Measurements

Runs all 7 empirical formulae against the OSSI wave gauge and compares
predicted vs measured wave heights.  Uses `compute_point_impact` to find
wake arrivals at the gauge, then applies each formula at the computed
lateral distance.

**Requires**: `ossi_xlsx`, `gauge_lon`, `gauge_lat` set in Cell 1.
Uses `df_vessel` from Cell 5 (re-derives block coefficients with `ossi_cb_method`).

In [ ]:
# === Cell 9a: OSSI comparison — Step 1 ===
# Prepare vessel data (re-derive Cb with ossi_cb_method if different)
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from aiswakepy.stages.wave_impact import compute_point_impact
from aiswakepy.models.kriebel  import compute_kriebel
from aiswakepy.models.pianc    import compute_pianc
from aiswakepy.models.bhowmik  import compute_bhowmik
from aiswakepy.models.gates    import compute_gates
from aiswakepy.models.blaauw   import compute_blaauw
from aiswakepy.models.sorensen import compute_sorensen
from aiswakepy.models.maynord  import compute_maynord
from aiswakepy.vessel.block_coeff import get_vessel_params_df

df_ossi_vessel = df_vessel.copy()
if ossi_cb_method != cb_method:
    df_ossi_vessel = get_vessel_params_df(df_ossi_vessel, method=ossi_cb_method)
    df_ossi_vessel["displacement_m3"] = (
        df_ossi_vessel["width"] * df_ossi_vessel["draught"]
        * df_ossi_vessel["length"] * 0.95 * df_ossi_vessel["block_coeff"]
    )

# Drop zero-draught records (unreliable AIS static data)
df_ossi_vessel = df_ossi_vessel[df_ossi_vessel["draught"] > 0].reset_index(drop=True)
print(f"Vessel records for OSSI comparison: {len(df_ossi_vessel)}")

In [ ]:
# === Cell 9b: OSSI comparison — Step 2 ===
# Find wake arrivals at the OSSI gauge
print(f"Finding wake arrivals at gauge ({gauge_lon}, {gauge_lat})...")
events = compute_point_impact(df_ossi_vessel, gauge_lon, gauge_lat, g=gravity)
print(f"  {len(events)} wake-arrival events")

if events.empty:
    raise RuntimeError("No wake events reached the gauge — check gauge coordinates and AIS data overlap.")

In [ ]:
# === Cell 9c: OSSI comparison — Step 3 ===
# Compute wave height with all 7 formulae
events["dist_perp"] = events["DistPerp_m"]

_formulae = {
    "Kriebel":  (compute_kriebel,  {"min_Froude_M": min_Froude_M, "max_Froude_M": max_Froude_M, "max_bf": max_bf}),
    "PIANC":    (compute_pianc,    {}),
    "Sorensen": (compute_sorensen, {}),
    "Maynord":  (compute_maynord,  {}),
    "Bhowmik":  (compute_bhowmik,  {}),
    "Gates":    (compute_gates,    {}),
    "Blaauw":   (compute_blaauw,   {}),
}

for label, (fn, kw) in _formulae.items():
    col = f"H_{label}"
    events[col] = fn(events, g=gravity, **kw).values
    n_valid = events[col].notna().sum() & (events[col] > 0).sum()
    print(f"  {label}: {n_valid} valid predictions")

In [ ]:
# === Cell 9d: OSSI comparison — Step 4 ===
# Load OSSI measurements
from aiswakepy.comparison import load_ossi

print(f"Loading OSSI measurements from {ossi_xlsx}...")
ossi = load_ossi(ossi_xlsx)
print(f"  {len(ossi)} OSSI wave events loaded")

In [ ]:
# === Cell 9e: OSSI comparison — Step 5 ===
# Match wake arrivals ↔ OSSI events
from aiswakepy.comparison import match_events

events["Hmax_measured"] = match_events(events["ArrivalTime"], ossi, event_window)
n_matched = events["Hmax_measured"].notna().sum()
print(f"  {n_matched} matched wake↔OSSI pairs (±{event_window} min window)")

In [ ]:
# === Cell 9f: OSSI comparison — Step 6 ===
# Setup colours and column map
from aiswakepy.comparison import COLOURS

pred_cols = {label: f"H_{label}" for label in _formulae}
print(f"Prediction columns: {list(pred_cols.keys())}")

In [ ]:
# === Cell 9g: OSSI comparison — Step 7 ===
# Time series plot — all predictions + measurements
from pathlib import Path
from aiswakepy.comparison import timeseries_plot

out = Path(output_dir) / "comparison"
out.mkdir(parents=True, exist_ok=True)

timeseries_plot(
    events, ossi, pred_cols,
    title="All empirical formulae vs OSSI measurements (arrival time at gauge)",
    out_path=out / "timeseries_all.png",
)
plt.show()

In [ ]:
# === Cell 9h: OSSI comparison — Step 8 ===
# Scatter plot — predicted vs measured (matched events only)
from aiswakepy.comparison import scatter_plot

scatter_plot(events, pred_cols, out / "scatter_predicted_vs_measured.png")
plt.show()

In [ ]:
# === Cell 9i: OSSI comparison — Step 9 ===
# Summary statistics for matched events
if n_matched > 0:
    stats_rows = []
    for label, col in pred_cols.items():
        valid = events[["Hmax_measured", col]].dropna()
        if valid.empty:
            continue
        meas = valid["Hmax_measured"].to_numpy()
        pred = valid[col].to_numpy()
        bias = np.mean(pred - meas)
        rmse = np.sqrt(np.mean((pred - meas) ** 2))
        r2 = np.corrcoef(meas, pred)[0, 1] ** 2 if len(meas) > 1 else float("nan")
        stats_rows.append({"Formula": label, "N": len(meas),
                           "Bias (m)": f"{bias:.4f}", "RMSE (m)": f"{rmse:.4f}",
                           "R²": f"{r2:.3f}"})
    df_stats = pd.DataFrame(stats_rows)
    print("\nSummary statistics (matched events):")
    display(df_stats)

    # Save matched events CSV
    matched_cols = ["DateTime", "ArrivalTime", "MMSI", "PropDist_m", "DistPerp_m",
                    "SOG", "VesselLength", "VesselWidth", "Side", "segment_id",
                    "Hmax_measured"] + list(pred_cols.values())
    out_cols = [c for c in matched_cols if c in events.columns]
    events[out_cols][events["Hmax_measured"].notna()].to_csv(out / "matched_events.csv", index=False)
    print(f"  saved matched_events.csv")
else:
    print("No matched events — try widening event_window or checking time alignment.")